# Hotel Booking Cancellation: Exploratory Data Analysis

This notebook profiles the Hotel Booking Demand dataset for the booking-creation prediction setting. The target is `is_canceled` (`1` = canceled, `0` = not canceled).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
data_candidates = [
    Path('data/raw/hotel_bookings.csv'),
    Path('../data/raw/hotel_bookings.csv'),
]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Download the Kaggle CSV to data/raw/hotel_bookings.csv before running this notebook.')

df = pd.read_csv(data_path, na_values=['NULL'])
print(f'Dataset path: {data_path}')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
required_columns = {'is_canceled', 'reservation_status', 'reservation_status_date'}
missing_required = required_columns.difference(df.columns)
assert not missing_required, f'Missing required columns: {missing_required}'
assert set(df['is_canceled'].dropna().unique()).issubset({0, 1})

display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print(f'Exact duplicate rows: {df.duplicated().sum():,}')
print('Duplicates are retained for now: the dataset has no booking identifier to distinguish duplicate records from legitimate identical bookings.')

In [ ]:
target_counts = df['is_canceled'].value_counts().sort_index()
target_rates = df['is_canceled'].value_counts(normalize=True).sort_index()
display(pd.DataFrame({'count': target_counts, 'rate': target_rates}))

ax = sns.countplot(data=df, x='is_canceled', hue='is_canceled', legend=False)
ax.set(title='Cancellation distribution', xlabel='is_canceled', ylabel='Bookings')
plt.show()

## Leakage and prediction-time feature audit

The primary model is defined at **booking creation**. The source dataset does not include field-level timestamps, so uncertain lifecycle fields are excluded conservatively.

In [ ]:
feature_audit = pd.DataFrame([
    ('is_canceled', 'Exclude', 'Prediction target.'),
    ('reservation_status', 'Exclude', 'Final outcome status; direct target leakage.'),
    ('reservation_status_date', 'Exclude', 'Final-status timing; direct target leakage.'),
    ('assigned_room_type', 'Exclude', 'May be assigned or changed after the booking is created.'),
    ('booking_changes', 'Exclude', 'Accumulates changes after the original booking.'),
    ('days_in_waiting_list', 'Exclude', 'Can accumulate during the reservation lifecycle.'),
    ('required_car_parking_spaces', 'Conditional', 'Use only if frozen at creation; otherwise exclude.'),
    ('total_of_special_requests', 'Conditional', 'Use only if frozen at creation; otherwise exclude.'),
    ('arrival/stay/guest/booking-history fields', 'Candidate', 'Available at creation if captured in the reservation record.'),
    ('reserved_room_type', 'Candidate', 'Requested room at creation; do not substitute assigned_room_type.'),
    ('adr', 'Candidate', 'Use only if it is the price quoted at booking time.'),
], columns=['feature_or_group', 'decision', 'reason'])
feature_audit

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.drop('is_canceled')
display(df[numeric_columns].describe().T)

correlations = (
    df[numeric_columns.tolist() + ['is_canceled']]
    .corr(numeric_only=True)['is_canceled']
    .drop('is_canceled')
    .sort_values(key=np.abs, ascending=False)
)
display(correlations.to_frame('correlation_with_is_canceled'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='is_canceled', y='lead_time', showfliers=False, ax=axes[0])
axes[0].set_title('Lead time by cancellation outcome')
sns.boxplot(data=df, x='is_canceled', y='adr', showfliers=False, ax=axes[1])
axes[1].set_title('ADR by cancellation outcome')
plt.tight_layout()

In [ ]:
categorical_columns = [
    'hotel', 'meal', 'market_segment', 'distribution_channel', 'deposit_type',
    'customer_type', 'reserved_room_type', 'is_repeated_guest'
]

for column in categorical_columns:
    rates = (
        df.groupby(column, dropna=False)['is_canceled']
        .agg(bookings='size', cancellation_rate='mean')
        .sort_values('cancellation_rate', ascending=False)
    )
    print(f'\nCancellation rate by {column}')
    display(rates)

In [ ]:
high_missingness = df.isna().mean().sort_values(ascending=False)
display(high_missingness[high_missingness.gt(0)].to_frame('missing_rate'))

high_cardinality = df.nunique(dropna=False).sort_values(ascending=False)
display(high_cardinality.to_frame('unique_values'))

print('Preprocessing implications: impute children with a train-fitted statistic; treat agent/company as categorical identifiers or use frequency encoding; avoid fitting any imputer, encoder, scaler, or outlier treatment on validation/test data.')

## Initial findings and next-stage decisions

- The dataset has 119,390 records, 32 columns, and a 37.0% cancellation rate; use stratified splits and optimize F1 for `is_canceled = 1`.
- `company`, `agent`, `country`, and `children` contain missing values and need train-only imputation decisions.
- `reservation_status` and `reservation_status_date` are direct leakage and must never enter the feature matrix.
- Exclude `assigned_room_type`, `booking_changes`, and `days_in_waiting_list` from the booking-creation model. Keep the two conditional fields out of the primary model until their collection time is verified.
- `lead_time`, deposit type, market segment, hotel type, repeat-guest status, special requests, and parking are associated with cancellation; association is not proof of valid availability, so the feature audit governs use.
- Exact duplicates are retained initially because there is no booking ID. The preprocessing stage should report a sensitivity comparison before deciding whether to remove them.

Next: build a train-only preprocessing pipeline and baseline models using the approved feature set.